## Named Entity Recognition (NER)




**Named Entity Recognition (NER)** is a subtask of Information Extraction (IE) that identifies and classifies key information (entities) in unstructured text into predefined categories. These categories can include names of people (`PERSON`), organizations (`ORG`), geographic locations (`GPE`), dates (`DATE`), monetary values (`MONEY`), and more.

### The spaCy Library
**spaCy** is a popular, high-performance open-source library in Python designed specifically for advanced Natural Language Processing (NLP). Unlike other academic toolkits, spaCy is built for production use-cases, providing fast, pre-trained statistical models and transition-based pipelines that can recognize entities out of the box with high accuracy.

### Why Case Sensitivity Matters
Named Entity Recognition models heavily rely on typographical features such as capitalization to detect entities. In English, names of people, brands, countries, and institutions are almost always capitalized.

* **Case-sensitive representation:** `Apple` is recognized as an organization (`ORG`).
* **Lowercase representation:** `apple` refers to the fruit and is typically ignored by the entity recognizer.

Without proper casing, a model loses key contextual clues, which significantly decreases its extraction recall and precision.

In [9]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 30.5 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [10]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [11]:
# Sample sentence
sentence = "Why Apple is looking at buying U.K. startup for $1 billion ?"
doc = nlp(sentence)

for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

Apple 4 9 ORG
U.K. 31 35 GPE
$1 billion 48 58 MONEY


In [12]:
# Check case sensitive
sentence = "Why apple is looking at buying U.K. startup for $1 billion ?"
doc = nlp(sentence)

for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

U.K. 31 35 GPE
$1 billion 48 58 MONEY


### SpaCy NER Labels and Descriptions
We will programmatically extract the entity labels supported by our loaded English model and use `spacy.explain` to build a DataFrame of their meanings.

In [13]:
import pandas as pd

# Get all NER labels supported by the loaded 'en_core_web_sm' model
ner_pipe = nlp.get_pipe('ner')
labels = sorted(list(ner_pipe.labels))

# Construct a list of labels and their official spaCy descriptions
ner_data = []
for label in labels:
    description = spacy.explain(label)
    ner_data.append({"NER Label": label, "Description": description})

# Create a DataFrame and display it
ner_df = pd.DataFrame(ner_data)
display(ner_df)

# Print the total number of labels to confirm we have at least 18
print(f"Total unique NER labels extracted: {len(ner_df)}")

,NER Label,Description
0,CARDINAL,Numerals that do not fall under another type
1,DATE,Absolute or relative dates or periods
2,EVENT,"Named hurricanes, battles, wars, sports events..."
3,FAC,"Buildings, airports, highways, bridges, etc."
4,GPE,"Countries, cities, states"
5,LANGUAGE,Any named language
6,LAW,Named documents made into laws.
7,LOC,"Non-GPE locations, mountain ranges, bodies of ..."
8,MONEY,"Monetary values, including unit"
9,NORP,Nationalities or religious or political groups


Total unique NER labels extracted: 18


### Adding Custom Computer Science (CSC) Entities
To add custom terms like `AI`, `ML`, `Python`, etc., and label them as `CSC` (Computer Science), we can use spaCy's `SpanRuler`. We will define 50 custom patterns and register them in the NLP pipeline.

In [14]:
# Define some Computer Science terms
csc_terms = [
    # Common terms
    "AI", "Artificial Intelligence", "ML", "Machine Learning", "Deep Learning",
    "NLP", "Natural Language Processing", "Computer Vision", "CV", "Data Analytics",
    "Python", "SQL", "HTML", "CSS", "Javascript", "Java", "C++", "Rust",
    "Go", "TypeScript", "Ruby", "Swift", "Kotlin", "PHP", "R", "Scala",
    "TensorFlow", "PyTorch", "Keras", "Scikit-Learn", "Pandas", "NumPy",
    "Docker", "Kubernetes", "AWS", "Azure", "GCP", "Hadoop", "Spark",
    "Git", "GitHub", "MongoDB", "PostgreSQL", "MySQL", "Redis", "GraphQL",
    "API", "Microservices", "DevOps", "CI/CD",
    # (AI, ML, DL, Networking, NLP)
    "OOP", "Propositional Logic", "Causal Reasoning", "Abductive Reasoning", "Game Theory",
    "Nash Equilibrium", "Bayesian Reasoning", "Supervised Learning", "Unsupervised Learning",
    "Overfitting", "Underfitting", "Reinforcement Learning", "DFS", "BFS", "Admissible Heuristics",
    "Minimax", "Alpha Beta Pruning", "Routers", "Virtual Machines", "Hybrid Cloud",
    "Public Cloud", "Private Cloud", "Community Cloud", "LLM", "IE", "NER", "POS Tagging",
    "Word Embedding", "Tokenization", "Linear Regression", "Logistic Regression", "SVM",
    "Naive Bayes", "LDA", "QDA", "Decision Trees", "Random Forest", "Gradient Boosting",
    "XGBoost", "AdaBoost", "Neural Networks", "MLP", "Regularization", "Perceptrons",
    "RNN", "CNN", "GAN", "LSTM", "GRU", "Dropout", "EarlyStopping",
    "Loss Function", "Activation Functions", "Sigmoid", "Softmax", "Epoch",
    "Batch Size", "Stochastic Gradient Descent", "Generative AI", "AI Agents"
]

# Remove existing span_ruler from the pipeline if it exists to avoid duplicate components
if "span_ruler" in nlp.pipe_names:
    nlp.remove_pipe("span_ruler")

# Configure a new SpanRuler to add all patterns as 'CSC' entities
ruler = nlp.add_pipe("span_ruler", before="ner")

# Add case-insensitive patterns for each CS term
patterns = [{"label": "CSC", "pattern": [{"LOWER": term.lower()}]} for term in csc_terms]
ruler.add_patterns(patterns)

print(f"Total custom CSC terms in the pipeline: {len(csc_terms)}")

Total custom CSC terms in the pipeline: 110


### Testing the Custom Entities
Let's test our updated pipeline with a sentence containing standard entities as well as our new custom `CSC` entities.

In [15]:
test_sentence = "While learning Python and SQL for Machine Learning, Alice used PyTorch on AWS to train her AI models."
doc = nlp(test_sentence)

print(f"Text: {test_sentence}\n")
print(f"{'Entity':<25} | {'Label':<10} | {'Description'}")
print("-" * 65)
for span in doc.spans.get("ruler", []):
    print(f"{span.text:<25} | {span.label_:<10} | Computer Science Term")
for ent in doc.ents:
    print(f"{ent.text:<25} | {ent.label_:<10} | {spacy.explain(ent.label_)}")

Text: While learning Python and SQL for Machine Learning, Alice used PyTorch on AWS to train her AI models.

Entity                    | Label      | Description
-----------------------------------------------------------------
Python                    | CSC        | Computer Science Term
SQL                       | CSC        | Computer Science Term
PyTorch                   | CSC        | Computer Science Term
AWS                       | CSC        | Computer Science Term
AI                        | CSC        | Computer Science Term
Python                    | PERSON     | People, including fictional
SQL                       | ORG        | Companies, agencies, institutions, etc.
Alice                     | PERSON     | People, including fictional
PyTorch                   | PERSON     | People, including fictional
AI                        | ORG        | Companies, agencies, institutions, etc.


### Why are `Python` and `PyTorch` tagged as `PERSON`, and `AI` as `ORG`?

When we look at the results of the default spaCy model (excluding our custom ruler), we notice some unexpected classifications:
* **`Python`** $\rightarrow$ `PERSON`
* **`PyTorch`** $\rightarrow$ `PERSON`
* **`AI`** $\rightarrow$ `ORG`

These misclassifications happen due to how statistical NER models operate:

1. **Lack of Domain-Specific Training Data:**
   The standard `en_core_web_sm` model is trained primarily on general-purpose datasets (such as OntoNotes, which consists of news, blogs, and broadcast conversations). In these corpora, technical terms, programming languages, and library names like "Python" and "PyTorch" are rare or entirely absent. Consequently, the model has to make its best guess based on the surrounding context.

2. **Contextual Clues and Capitalization:**
   * **`Python` and `PyTorch` as `PERSON`:** In English grammar, words that are capitalized and function as nouns in subject or object positions (e.g., *"learning [Name]"* or *"[Name] used..."*) highly resemble proper nouns like names of people. Without knowing that "Python" refers to a programming language, the model's weights assign a high probability to the `PERSON` tag because human names frequently appear in these structural patterns.
   * **`AI` as `ORG`:** Capitalized acronyms (especially 2-3 letter abbreviations) in news articles are overwhelmingly organizations (like *UN, EU, IMF, GM*). The statistical model associates capitalized short tokens with the `ORG` label due to patterns it observed in its training data.

This highlights exactly why adding custom pipeline components like **`SpanRuler`** is so crucial when working with specialized domains such as Computer Science and Tech!

### Interactive File Uploader & NER Visualizer
The following cell generates an interactive HTML and JavaScript interface. You can upload any `.txt` or `.md` file, and it will send the content to the Python backend to perform Named Entity Recognition with both standard labels and our custom `CSC` (Computer Science) spans.

In [16]:
import json
import spacy
from IPython.display import HTML, display
from google.colab import output

# HTML template for file upload (accepting .txt and .md), analysis, and custom colored entity rendering
html_interface = """
<!DOCTYPE html>
<html>
<head>
    <style>
        .ner-container {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            padding: 20px;
            border: 1px solid #ddd;
            border-radius: 8px;
            background-color: #f9f9f9;
            max-width: 800px;
            margin: auto;
        }
        .upload-section {
            border: 2px dashed #007bff;
            padding: 20px;
            text-align: center;
            background-color: #fff;
            cursor: pointer;
            border-radius: 6px;
            margin-bottom: 20px;
        }
        .upload-section:hover {
            background-color: #f1f7ff;
        }
        .result-box {
            background: white;
            border: 1px solid #ccc;
            padding: 15px;
            border-radius: 6px;
            white-space: pre-wrap;
            line-height: 2;
            display: none;
        }
        .entity {
            padding: 2px 6px;
            border-radius: 4px;
            font-weight: bold;
            margin: 0 2px;
            display: inline-block;
            line-height: 1.2;
        }
        .ent-csc { background-color: #ffcccb; color: #900; } /* Light red */
        .ent-person { background-color: #ffe57f; color: #b78103; } /* Gold */
        .ent-org { background-color: #b3e5fc; color: #0277bd; } /* Blue */
        .ent-gpe { background-color: #c8e6c9; color: #2e7d32; } /* Green */
        .ent-other { background-color: #e1bee7; color: #6a1b9a; } /* Purple */

        .label-tag {
            font-size: 0.75em;
            font-weight: normal;
            text-transform: uppercase;
            margin-left: 5px;
            padding: 1px 4px;
            border-radius: 3px;
            background: rgba(0,0,0,0.1);
        }
    </style>
</head>
<body>
    <div class="ner-container">
        <h3>Custom & Standard NER File Analyzer</h3>
        <p>Upload a <code>.txt</code> or <code>.md</code> file to extract its contents and run the NER parser.</p>

        <div class="upload-section" onclick="document.getElementById('fileInput').click()">
            <strong>Click here to select a .txt or .md file</strong>
            <input type="file" id="fileInput" accept=".txt,.md" style="display:none;" onchange="handleFile(this.files)">
        </div>

        <div id="result" class="result-box"></div>
    </div>

    <script>
        function handleFile(files) {
            if (files.length === 0) return;
            const file = files[0];
            const reader = new FileReader();

            reader.onload = function(e) {
                const text = e.target.result;
                document.getElementById('result').style.display = 'block';
                document.getElementById('result').innerHTML = 'Processing document...';

                // Call the Python backend function
                google.colab.kernel.invokeFunction('notebook.process_text', [text], {});
            };
            reader.readAsText(file);
        }

        // Callback registered to populate the analyzed text with markup
        window.updateResults = function(htmlContent) {
            const resDiv = document.getElementById('result');
            resDiv.innerHTML = htmlContent;
        }
    </script>
</body>
</html>
"""

def process_text(text):
    doc = nlp(text)

    # Track entities and their start/end indices
    spans_to_render = []

    # Gather custom ruler spans
    for span in doc.spans.get("ruler", []):
        spans_to_render.append((span.start_char, span.end_char, span.text, "CSC"))

    # Gather standard entities (excluding overlaps that are already marked as CSC to prevent nested nesting)
    csc_intervals = [(s[0], s[1]) for s in spans_to_render]

    for ent in doc.ents:
        # Simple non-overlap check
        overlap = any(not (ent.end_char <= start or ent.start_char >= end) for start, end in csc_intervals)
        if not overlap:
            spans_to_render.append((ent.start_char, ent.end_char, ent.text, ent.label_))

    # Sort spans by starting index to build markup sequentially
    spans_to_render = sorted(spans_to_render, key=lambda x: x[0])

    # Reconstruct the text with HTML markers
    html_output = ""
    last_idx = 0

    for start, end, original_text, label in spans_to_render:
        if start < last_idx:
            continue  # Safety check for overlapping entities

        # Add plain text leading up to entity
        html_output += text[last_idx:start]

        # Classify color groups
        cls = "ent-other"
        if label == "CSC": cls = "ent-csc"
        elif label == "PERSON": cls = "ent-person"
        elif label == "ORG": cls = "ent-org"
        elif label == "GPE": cls = "ent-gpe"

        # Wrap entity in styled HTML tags
        html_output += f'<span class="entity {cls}">{original_text}<span class="label-tag">{label}</span></span>'
        last_idx = end

    html_output += text[last_idx:]

    # Pass the formatted HTML back to JavaScript
    output.eval_js(f"window.updateResults({json.dumps(html_output)})")

# Register the Python callback function so JS can invoke it
output.register_callback('notebook.process_text', process_text)

# Display the HTML uploader
display(HTML(html_interface))